In [1]:
import os
os.chdir("../../..")

In [2]:
import torch
from models.DeepCNN import DeepCNN
from models.RamanFormer import RamanFormer
from models.RamanNet import RamanNet
from models.SANet import ScaleAdaptiveNet
from models.transformer import ViT
from torch.utils.data import Dataset,DataLoader
import pickle
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

In [3]:
class Pharma_dataset(Dataset):
  def __init__(self,path):
    """
    path is a string containing the path to the pkl dataset
    """
    super().__init__()   
    #X is a list with each element of the list containing a 1024 time series data 
    #y is a list with containing the name of the chemical corresponding to X
    self.y, self.X = pickle.load(open(path, 'rb'))
    names = sorted(list(set(self.y)))
    self.mapping = {names[i]:i for i in range(len(names))}   #Maps each material name to a number

  def __len__(self):
    return len(self.y)

  def __getitem__(self,index):
    data = torch.Tensor(self.X[index]) #of shape (1,1024)
    data = (data-data.min())/(data.max()-data.min())
    label = self.mapping[self.y[index]]
    return data,label

In [4]:
def test_f1(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [5]:
def test_f1_RamanNet(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred,_ = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [6]:
def test_f1_ML(model,test_set):
    X_test = [test_set[i][0].squeeze() for i in range(len(test_set))]
    y_test = [test_set[i][1] for i in range(len(test_set))]

    all_preds = model.predict(X_test)
    all_preds = np.array(all_preds)
    all_targets = np.array(y_test)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    return accuracy, precision, recall, f1

In [7]:
def test_f1_wrapper(model1,model2,model3,model4,model5,device,test_dataloader,model_type):

    if model_type=="DL":
        all_acc1, _, _, all_f11 = test_f1(model1,device,test_dataloader)
        all_acc2, _, _, all_f12 = test_f1(model2,device,test_dataloader)
        all_acc3, _, _, all_f13 = test_f1(model3,device,test_dataloader)
        all_acc4, _, _, all_f14 = test_f1(model4,device,test_dataloader)
        all_acc5, _, _, all_f15 = test_f1(model5,device,test_dataloader)

    elif model_type=="RamanNet":
        all_acc1, _, _, all_f11 = test_f1_RamanNet(model1,device,test_dataloader)
        all_acc2, _, _, all_f12 = test_f1_RamanNet(model2,device,test_dataloader)
        all_acc3, _, _, all_f13 = test_f1_RamanNet(model3,device,test_dataloader)
        all_acc4, _, _, all_f14 = test_f1_RamanNet(model4,device,test_dataloader)
        all_acc5, _, _, all_f15 = test_f1_RamanNet(model5,device,test_dataloader)
    
    else:
        all_acc1, _, _, all_f11 = test_f1_ML(model1,test_dataloader.dataset)
        all_acc2, _, _, all_f12 = test_f1_ML(model2,test_dataloader.dataset)
        all_acc3, _, _, all_f13 = test_f1_ML(model3,test_dataloader.dataset)
        all_acc4, _, _, all_f14 = test_f1_ML(model4,test_dataloader.dataset)
        all_acc5, _, _, all_f15 = test_f1_ML(model5,test_dataloader.dataset)      

    all_acc = np.array([all_acc1,all_acc2,all_acc3,all_acc4,all_acc5])
    all_f1 = np.array([all_f11,all_f12,all_f13,all_f14,all_f15])

    print(f"{round(all_acc.mean(),2)}\\% (\u00B1 {round(all_acc.std(),2)})& {round(all_f1.mean(),4)} (\u00B1 {round(all_f1.std(),4)}) \\\\")

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [9]:
test_set = Pharma_dataset("datasets/Pharma/Pharma_test.pkl")
test_loader = DataLoader(test_set, batch_size=32, num_workers=8,shuffle=False)
print(f"The number of elements in the test set is {len(test_loader.dataset)}")

The number of elements in the test set is 704


In [10]:
deepcnn1= DeepCNN(num_classes=32).to(device)
deepcnn1.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_deepcnn_0_3_98.04_.pt",weights_only=True))

deepcnn2= DeepCNN(num_classes=32).to(device)
deepcnn2.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_deepcnn_1_6_96.79_.pt",weights_only=True))

deepcnn3= DeepCNN(num_classes=32).to(device)
deepcnn3.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_deepcnn_2_13_98.22_.pt",weights_only=True))

deepcnn4= DeepCNN(num_classes=32).to(device)
deepcnn4.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_deepcnn_3_5_97.33_.pt",weights_only=True))

deepcnn5= DeepCNN(num_classes=32).to(device)
deepcnn5.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_deepcnn_4_2_98.04_.pt",weights_only=True))

RamanFormer1 = RamanFormer(num_classes=32).to(device)
RamanFormer1.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanFormer_0_27_98.04_.pt",weights_only=True))

RamanFormer2 = RamanFormer(num_classes=32).to(device)
RamanFormer2.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanFormer_1_32_97.15_.pt",weights_only=True))

RamanFormer3 = RamanFormer(num_classes=32).to(device)
RamanFormer3.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanFormer_2_22_97.5_.pt",weights_only=True))

RamanFormer4 = RamanFormer(num_classes=32).to(device)
RamanFormer4.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanFormer_3_27_97.15_.pt",weights_only=True))

RamanFormer5 = RamanFormer(num_classes=32).to(device)
RamanFormer5.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanFormer_4_33_98.04_.pt",weights_only=True))

RamanNet1 = RamanNet(num_classes=32).to(device)
RamanNet1.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanNet_0_12_98.04_.pt",weights_only=True))

RamanNet2 = RamanNet(num_classes=32).to(device)
RamanNet2.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanNet_1_11_97.68_.pt",weights_only=True))

RamanNet3 = RamanNet(num_classes=32).to(device)
RamanNet3.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanNet_2_8_96.97_.pt",weights_only=True))

RamanNet4 = RamanNet(num_classes=32).to(device)
RamanNet4.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanNet_3_19_97.15_.pt",weights_only=True))

RamanNet5 = RamanNet(num_classes=32).to(device)
RamanNet5.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_RamanNet_4_8_97.86_.pt",weights_only=True))

SANet1 = ScaleAdaptiveNet(num_classes=32).to(device)
SANet1.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_SANet_0_6_99.82_.pt",weights_only=True))

SANet2 = ScaleAdaptiveNet(num_classes=32).to(device)
SANet2.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_SANet_1_20_100.0_.pt",weights_only=True))

SANet3 = ScaleAdaptiveNet(num_classes=32).to(device)
SANet3.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_SANet_2_10_99.64_.pt",weights_only=True))

SANet4 = ScaleAdaptiveNet(num_classes=32).to(device)
SANet4.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_SANet_3_12_99.64_.pt",weights_only=True))

SANet5 = ScaleAdaptiveNet(num_classes=32).to(device)
SANet5.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_SANet_4_16_100.0_.pt",weights_only=True))

transformer1 = ViT(patch_size=128,p=0.1,num_classes=32).to(device)
transformer1.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_transformer_0_32_97.68_.pt",weights_only=True))

transformer2 = ViT(patch_size=128,p=0.1,num_classes=32).to(device)
transformer2.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_transformer_1_35_96.26_.pt",weights_only=True))

transformer3 = ViT(patch_size=128,p=0.1,num_classes=32).to(device)
transformer3.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_transformer_2_30_97.15_.pt",weights_only=True))

transformer4 = ViT(patch_size=128,p=0.1,num_classes=32).to(device)
transformer4.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_transformer_3_39_96.97_.pt",weights_only=True))

transformer5 = ViT(patch_size=128,p=0.1,num_classes=32).to(device)
transformer5.load_state_dict(torch.load("results/final_multi_run/trained_models/Pharma_transformer_4_35_97.68_.pt",weights_only=True))


with open('results/final_multi_run/trained_models/Pharma_RandomForest_0_99.47_.pkl', 'rb') as file:
    randomforest1 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_RandomForest_1_99.64_.pkl', 'rb') as file:
    randomforest2 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_RandomForest_2_99.29_.pkl', 'rb') as file:
    randomforest3 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_RandomForest_3_99.11_.pkl', 'rb') as file:
    randomforest4 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_RandomForest_4_99.47_.pkl', 'rb') as file:
    randomforest5 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_SVC_0_97.68_.pkl', 'rb') as file:
    svc1 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_SVC_1_96.61_.pkl', 'rb') as file:
    svc2 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_SVC_2_97.86_.pkl', 'rb') as file:
    svc3 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_SVC_3_97.15_.pkl', 'rb') as file:
    svc4 = pickle.load(file)

with open('results/final_multi_run/trained_models/Pharma_SVC_4_97.68_.pkl', 'rb') as file:
    svc5 = pickle.load(file)

In [11]:
test_f1_wrapper(deepcnn1,deepcnn2,deepcnn3,deepcnn4,deepcnn5,device,test_loader,model_type="DL")

100%|██████████| 22/22 [00:00<00:00, 88.27it/s]

97.02\% (± 0.16)& 0.965 (± 0.0057) \\


In [12]:
test_f1_wrapper(RamanFormer1,RamanFormer2,RamanFormer3,RamanFormer4,RamanFormer5,device,test_loader,model_type="DL")

100%|██████████| 22/22 [00:00<00:00, 79.16it/s]

96.85\% (± 0.11)& 0.9628 (± 0.0048) \\


In [13]:
test_f1_wrapper(RamanNet1,RamanNet2,RamanNet3,RamanNet4,RamanNet5,device,test_loader,model_type="RamanNet")

100%|██████████| 22/22 [00:00<00:00, 63.87it/s]

96.73\% (± 0.2)& 0.9593 (± 0.0049) \\


In [14]:
test_f1_wrapper(SANet1,SANet2,SANet3,SANet4,SANet5,device,test_loader,model_type="DL")

100%|██████████| 22/22 [00:00<00:00, 58.76it/s]

99.63\% (± 0.21)& 0.9963 (± 0.0021) \\


In [15]:
test_f1_wrapper(transformer1,transformer2,transformer3,transformer4,transformer5,device,test_loader,model_type="DL")

100%|██████████| 22/22 [00:00<00:00, 54.76it/s]


96.14\% (± 0.36)& 0.9566 (± 0.0029) \\


In [16]:
test_f1_wrapper(randomforest1,randomforest2,randomforest3,randomforest4,randomforest5,device,test_loader,model_type="ML")

98.66\% (± 0.17)& 0.9867 (± 0.0017) \\


In [17]:
test_f1_wrapper(svc1,svc2,svc3,svc4,svc5,device,test_loader,model_type="ML")

97.16\% (± 0.16)& 0.968 (± 0.0016) \\
